## **Bibliotecas**

In [2]:
#%pip install pydantic[email]

from datetime import date, datetime
from typing import List, Optional
from uuid import uuid4
from enum import StrEnum, auto
import json

from pydantic import (
    BaseModel,
    EmailStr,
    Field, 
    field_serializer,
    UUID4,
)

from fastapi import FastAPI
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient

## **FastAPI**

In [3]:
app = FastAPI()

## **Classes**

In [ ]:
class NivelRisco(StrEnum):
    """
    Define os níveis de severidade das lesões bucais detectadas.
    """
    
    BAIXO = auto()
    MODERADO = auto()
    ALTO = auto()
    CRITICO = auto()

In [ ]:
class Consulta(BaseModel):
    """
    Representa o registro clínico de uma avaliação de triagem.
    Esta classe armazena as observações do profissional de saúde e o 
    resultado do nível de risco para monitoramento do câncer de boca.
    """

    model_config = {
        "extra": "forbid",
    }

    id_consulta: UUID4 = Field(
        default_factory=uuid4,
        description="ID Único da consulta"
    )

    data_consulta: date = Field(
        default_factory=date.today,
        description="Data da realização da consulta"
    )

    observacoes: str = Field(
        ...,
        description="Relato do profissional de saúde"
    )

    risco_detectado: NivelRisco = Field(
        default=NivelRisco.BAIXO,
        description="Nível de risco de câncer de boca detectado"
    )


    @field_serializer("id_consulta", when_used="json")
    def serialize_id(self, id_consulta: UUID4) -> str:
        """
        Converte o UUID em string para compatibilidade com o TestClient.
        """

        return str(id_consulta)
    
    @field_serializer("data_consulta", when_used="json")
    def serialize_date(self, data_consulta: date) -> str:
        """
        Converte a data no formato ISO para compatibilidade com o TestClient.
        """

        return data_consulta.isoformat()

In [ ]:
class Paciente(BaseModel):
    """
    Armazena os dados do paciente, fatores de risco (tabagismo/alcoolismo) 
    e o histórico clínico completo para prevenção de câncer de boca.
    """

    model_config = {
        "extra": "forbid",
    }

    # Simulação de banco de dados em memória
    __pacientes__ = [] 
    
    id: UUID4 = Field(
        default_factory=uuid4,
        description="ID Único do Paciente"
    )

    nome_completo: str = Field(
        ..., description="Nome completo do Paciente"
    )

    email: EmailStr = Field(
        ...,description="Email do Paciente"
    )

    data_nascimento: date = Field(
        ..., description="Data de nascimento do Paciente"
    )
    
    # Fatores de risco primários para câncer bucal
    fumante: bool = Field(
        default=False,
        description="Indica se o paciente possui hábito de tabagismo"
    )
    
    consumo_alcool: bool = Field(
        default=False,
        description="Indica se o paciente consome bebidas alcoólicas"
    )
    
    historico_consultas: List[Consulta] = Field(
        default_factory=list,
        description="Lista de consultas realizadas"
    )


    @field_serializer("id", when_used="json")
    def serialize_id(self, id: UUID4) -> str:
        """
        Converte o UUID em string para compatibilidade com o TestClient.
        """
        
        return str(id)

    @field_serializer("data_nascimento", when_used="json")
    def serialize_date(self, data_nascimento: date) -> str:
        """
        Converte a data no formato ISO para compatibilidade com o TestClient.
        """ 

        return data_nascimento.isoformat()

## **Endpoints**

In [ ]:
@app.get("/pacientes", response_model=list[Paciente])
async def get_pacientes() -> list[Paciente]:
    """
    Retorna a lista de todos os pacientes cadastrados.
    """
    
    return list(Paciente.__pacientes__)

In [ ]:
@app.post("/pacientes", response_model=Paciente)
async def create_paciente(paciente: Paciente):
    """
    Cria um novo paciente no sistema e realiza a validação automática de dados.
    """
    
    Paciente.__pacientes__.append(paciente)
    return paciente

In [ ]:
@app.get("/pacientes/{paciente_id}", response_model=Paciente)
async def get_paciente(paciente_id: UUID4) -> Paciente | JSONResponse:
    """
    Busca um paciente pelo seu ID.
    """
    
    try:
        return next((paciente for paciente in Paciente.__pacientes__ if paciente.id == paciente_id))
    except StopIteration:
        return JSONResponse(status_code=404, content={"message": "Paciente não encontrado"})

In [ ]:
@app.get("/pacientes/{paciente_id}/consultas", response_model=List[Consulta])
async def get_consultas_paciente(paciente_id: UUID4):
    """
    Retorna a lista de todas as consultas de um paciente específico.
    """
    
    try: 
        paciente = next((paciente for paciente in Paciente.__pacientes__ if paciente.id == paciente_id))
        return paciente.historico_consultas
    except StopIteration:
        return JSONResponse(status_code=404, content={"message": "Paciente não encontrado"})

In [ ]:
@app.post("/pacientes/{paciente_id}/consultas", response_model=Consulta)
async def create_consulta_paciente(paciente_id: UUID4, consulta: Consulta):
    """
    Cria uma nova consulta para um paciente existente.
    """
    
    try: 
        paciente = next((paciente for paciente in Paciente.__pacientes__ if paciente.id == paciente_id))
        paciente.historico_consultas.append(consulta)
        return consulta
    except StopIteration:
        return JSONResponse(status_code=404, content={"message": "Paciente não encontrado"})

## **Main**

In [ ]:
with TestClient(app) as client:
    # Reseta a lista de pacientes
    Paciente.__pacientes__ = []

    # Cria 5 pacientes
    for i in range(5):
        response = client.post(
            "/pacientes",
            json={
                "nome_completo": f"Paciente {i}", 
                "email": f"teste{i}@sobrevidas.com.br",
                "data_nascimento": "2004-08-26",
                "fumante": False,
                "consumo_alcool": False
            },
        )
        assert response.status_code == 200
        assert response.json()["nome_completo"] == f"Paciente {i}", (
            f"O nome do paciente deveria ser Paciente {i}"
        )
        assert response.json()["id"], "O paciente deveria ter um ID"

        # Validação do Modelo Pydantic para garantir que possuem os tipos corretos
        paciente = Paciente.model_validate(response.json())

        assert str(paciente.id) == response.json()["id"], "O ID deve ser o mesmo"
        assert paciente.historico_consultas == [], "O histórico deve iniciar vazio"


    # Verificando se todos os 5 pacientes foram listados
    response = client.get("/pacientes")
    assert response.status_code == 200, "O código de resposta deve ser 200"
    assert len(response.json()) == 5, "Deveria haver 5 pacientes na lista"


    # Testando a criação de um paciente para monitoramento
    response = client.post(
        "/pacientes",
        json={
            "nome_completo": "Paciente 5", 
            "email": "paciente5@sobrevidas.com.br",
            "data_nascimento": "2004-08-26",
            "fumante": True
        }
    )
    assert response.status_code == 200

    assert response.json()["id"], (
        "O paciente deveria ter um ID"
    )
    assert response.json()["nome_completo"] == "Paciente 5", (
        "O nome do paciente deveria ser Paciente 5"
    )
    assert response.json()["email"] == "paciente5@sobrevidas.com.br", (
        "O email do paciente deveria ser paciente5@sobrevidas.com.br"
    )
    assert response.json()["data_nascimento"] == "2004-08-26", (
        "A data de nascimento do paciente deveria ser 2004-08-26"
    )
    assert response.json()["fumante"] == True, (
        "Este paciente deve ser marcado como fumante"
    )

    paciente = Paciente.model_validate(response.json())
    assert str(paciente.id) == response.json()["id"], "O ID deve ser o mesmo"
    assert paciente.historico_consultas == [], "O histórico deve iniciar vazio"


    # Testando a busca por um paciente específico
    response = client.get(f"/pacientes/{paciente.id}")
    assert response.status_code == 200
    assert response.json()["nome_completo"] == "Paciente 5", (
        "Deve ser o nome do paciente recém-criado"
    )


    # Testando a busca por um paciente que não existe
    response = client.get(f"/pacientes/{uuid4()}")
    assert response.status_code == 404
    assert response.json()["message"] == "Paciente não encontrado", (
        "Deveria retornar mensagem de erro para ID inexistente"
    )


    # Testando um Email inválido
    response = client.post(
        "/pacientes",
        json={
            "nome_completo": "Paciente Erro", 
            "email": "email_invalido", 
            "data_nascimento": "2004-08-26"
        }
    )
    assert response.status_code == 422, "O endereço de email deve ser invalidado pelo Pydantic"

    print("✅ Todos os testes de fluxo do SobreVidas passaram com sucesso!")

✅ Todos os testes de fluxo do SobreVidas passaram com sucesso!
